In [24]:

import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from simulation_function import *
from simulation_function_euler import *


In [25]:
'''
## param格式:
param = {
    "m1": 1.0,  # 摆球1质量 (kg)
    "m2": 1.0,  # 摆球2质量 (kg)
    "l1": 1.0,  # 摆绳1长度 (m)
    "l2": 1.0,  # 摆绳2长度 (m)
    "g": 9.81,  # 重力加速度 (m/s^2)
    "dt": 0.01,  # 时间步长 (s)
    "duration": 25.0,  # 总模拟时间 (s)
    "angle_mode": "DEG",  # 模式选择: 'DEG' (角度) 或 'RAD' (弧度)
    "theta1_0": 90.0,  # 初始角度1
    "theta2_0": 90.0002,  # 初始角度2
    "w1_0": 0.0,  # 初始角速度1 (无论模式，建议设为0)
    "w2_0": 0.0,  # 初始角速度2
}

"""

simulation function: 
simulate_double_pendulum(para = params)

"""
'''

'\n## param格式:\nparam = {\n    "m1": 1.0,  # 摆球1质量 (kg)\n    "m2": 1.0,  # 摆球2质量 (kg)\n    "l1": 1.0,  # 摆绳1长度 (m)\n    "l2": 1.0,  # 摆绳2长度 (m)\n    "g": 9.81,  # 重力加速度 (m/s^2)\n    "dt": 0.01,  # 时间步长 (s)\n    "duration": 25.0,  # 总模拟时间 (s)\n    "angle_mode": "DEG",  # 模式选择: \'DEG\' (角度) 或 \'RAD\' (弧度)\n    "theta1_0": 90.0,  # 初始角度1\n    "theta2_0": 90.0002,  # 初始角度2\n    "w1_0": 0.0,  # 初始角速度1 (无论模式，建议设为0)\n    "w2_0": 0.0,  # 初始角速度2\n}\n\n"""\n\nsimulation function: \nsimulate_double_pendulum(para = params)\n\n"""\n'

In [26]:
import pandas as pd
import numpy as np

In [27]:
param = {
    "m1": 1.0,  # 摆球1质量 (kg)
    "m2": 1.0,  # 摆球2质量 (kg)
    "l1": 1.0,  # 摆绳1长度 (m)
    "l2": 1.0,  # 摆绳2长度 (m)
    "g": 9.81,  # 重力加速度 (m/s^2)
    "dt": 0.01,  # 时间步长 (s)
    "duration": 15.0,  # 总模拟时间 (s)
    "angle_mode": "DEG",  # 模式选择: 'DEG' (角度) 或 'RAD' (弧度)
    "theta1_0": 90.0,  # 初始角度1
    "theta2_0": 90.0,  # 初始角度2
    "w1_0": 0.0,  # 初始角速度1 (无论模式，建议设为0)
    "w2_0": 0.0,  # 初始角速度2
}

In [28]:
#test = simulate_double_pendulum(param)
#test

In [29]:
#命名格式：run01/run_01_data_001_RK4_0.01_10s.csv

In [30]:
import tqdm
import csv

### 开始写逻辑

In [31]:
import pandas as pd
import tqdm
from pathlib import Path

# 1. 基础配置
run_number = "02"
folder_path = Path(f"run{run_number}")
folder_path.mkdir(exist_ok=True)

# 你的 10 个实验步长梯度
time_steps = [0.2, 0.1, 0.05, 0.02, 0.01, 0.0075, 0.005, 0.002, 0.001, 0.0005, 0.0002, 0.0001]
# 2 种核心算法对决
methods = ["RK4", "Euler"]

# 【核心前置条件】既然这次研究步长，必须死死固定住初始角度！
# 假设固定为你的基准角度，比如 theta1 = 90.0, theta2 = 90.0
param["theta1_0"] = 90.0
param["theta2_0"] = 90.0  # 不再变化

# 自动获取总模拟时间（用于文件名，比如 "25s"）
duration_str = f"{int(param['duration'])}s"

# 计算总任务数：2 种方法 * 10 个步长 = 20 次实验
total_experiments = len(methods) * len(time_steps)

# 2. 开始双重嵌套循环（用 tqdm 监控这 20 次模拟）
# 使用 itertools.product 或者简单的双循环，这里用计数器 num 来给文件编号
num_counter = 1

with tqdm.tqdm(total=total_experiments, desc="Running Step/Method Comparison") as pbar:
    for method in methods:
        for dt in time_steps:
            
            # 动态更新当前实验的参数
            param["dt"] = dt
            
            # 格式化文件编号（比如 001, 002...）
            num_str = f"{num_counter:03d}"
            
            # 构造全新的命名格式：run01/run_01_data_001_RK4_0.0100_25s.csv
            # dt 用 :.4f 确保在变成 0.0005 时不会因为浮点数精度变成一大串奇怪的数字
            filename = f"run_{run_number}_data_{num_str}_{method}_{dt:.4f}_{duration_str}.parquet"
            path = folder_path / filename
            
            # 运行模拟（将 method 传入你的求解器）
            # 提示：如果你的函数不支持 method 参数，可以改成：param["method"] = method，然后传 param
            if method == 'RK4':
                temp_storage = simulate_double_pendulum(param, energy = True)
            elif method == 'Euler':
                temp_storage = simulate_double_pendulum_euler(param, energy = True)
            
            # 写入对应的 CSV 文件
        df_out = pd.DataFrame(temp_storage, columns=['theta_1', 'theta_2', 'energy'])
        df_out.to_parquet(path, index=False)

print("\n🎉 所有算法与步长对比实验数据已完美落地！")

Running Step/Method Comparison:  50%|█████     | 12/24 [00:18<00:41,  3.49s/it]/Users/huangwenqin/Desktop/AIF+CALC+SSF PROJs/CALC_EOCPROJ/python_code/simulation_function_euler.py:73: RuntimeWarning: overflow encountered in scalar power
  
/Users/huangwenqin/Desktop/AIF+CALC+SSF PROJs/CALC_EOCPROJ/python_code/simulation_function_euler.py:74: RuntimeWarning: overflow encountered in scalar power
  results = []
/Users/huangwenqin/Desktop/AIF+CALC+SSF PROJs/CALC_EOCPROJ/python_code/simulation_function_euler.py:75: RuntimeWarning: overflow encountered in scalar multiply
  for i in range(num_steps):
/Users/huangwenqin/Desktop/AIF+CALC+SSF PROJs/CALC_EOCPROJ/python_code/simulation_function_euler.py:57: RuntimeWarning: overflow encountered in scalar power
  - 2 * np.sin(delta) * m2 * (w2**2 * l2 + w1**2 * l1 * np.cos(delta))
/Users/huangwenqin/Desktop/AIF+CALC+SSF PROJs/CALC_EOCPROJ/python_code/simulation_function_euler.py:57: RuntimeWarning: invalid value encountered in scalar add
  - 2 * np.s


🎉 所有算法与步长对比实验数据已完美落地！
